In [ ]:
import pandas as pd
import numpy as np
from core.cleaning import detect_stale_prices

Build a realistic dirty dataset first: a DataFrame of daily close prices for two tickers over 10 days that has at least one of each of the following planted in it:

* A missing date (gap in the date index)
* A NaN value
* A stale price (same value repeated across consecutive days)
* An obvious bad print (price that's clearly wrong — use your judgment on what that looks like)
* Set a datetime index

Clean starting data set with random seeding

In [44]:
dates = pd.date_range('2026-05-08', periods=11, freq='B')

df = []

for ticker, (low, high) in {'CL': (75, 85), 'NG': (2.0, 3.0)}.items():
    df.append(pd.DataFrame({'date': dates, 'ticker': ticker, 
                            'close': np.random.uniform(low, high, 
                                                       size=len(dates))}))

df = pd.concat(df).set_index('date').sort_index()

print(df)

           ticker      close
date                        
2026-05-08     CL  82.835803
2026-05-08     NG   2.812686
2026-05-11     CL  79.417698
2026-05-11     NG   2.290038
2026-05-12     CL  81.152721
2026-05-12     NG   2.110996
2026-05-13     CL  76.231733
2026-05-13     NG   2.462420
2026-05-14     CL  77.441923
2026-05-14     NG   2.639634
2026-05-15     CL  80.390492
2026-05-15     NG   2.835200
2026-05-18     CL  75.856269
2026-05-18     NG   2.089650
2026-05-19     CL  80.859832
2026-05-19     NG   2.502811
2026-05-20     NG   2.037123
2026-05-20     CL  79.223156
2026-05-21     NG   2.988673
2026-05-21     CL  76.359693
2026-05-22     CL  84.626354
2026-05-22     NG   2.015269


* A missing date (gap in the date index)


In [45]:
df = df.drop(pd.Timestamp('2026-05-20'))
print(df)

           ticker      close
date                        
2026-05-08     CL  82.835803
2026-05-08     NG   2.812686
2026-05-11     CL  79.417698
2026-05-11     NG   2.290038
2026-05-12     CL  81.152721
2026-05-12     NG   2.110996
2026-05-13     CL  76.231733
2026-05-13     NG   2.462420
2026-05-14     CL  77.441923
2026-05-14     NG   2.639634
2026-05-15     CL  80.390492
2026-05-15     NG   2.835200
2026-05-18     CL  75.856269
2026-05-18     NG   2.089650
2026-05-19     CL  80.859832
2026-05-19     NG   2.502811
2026-05-21     NG   2.988673
2026-05-21     CL  76.359693
2026-05-22     CL  84.626354
2026-05-22     NG   2.015269


* A NaN value


In [46]:
df.loc[[pd.Timestamp('2026-05-18')], ['close']] = None
print(df)

           ticker      close
date                        
2026-05-08     CL  82.835803
2026-05-08     NG   2.812686
2026-05-11     CL  79.417698
2026-05-11     NG   2.290038
2026-05-12     CL  81.152721
2026-05-12     NG   2.110996
2026-05-13     CL  76.231733
2026-05-13     NG   2.462420
2026-05-14     CL  77.441923
2026-05-14     NG   2.639634
2026-05-15     CL  80.390492
2026-05-15     NG   2.835200
2026-05-18     CL        NaN
2026-05-18     NG        NaN
2026-05-19     CL  80.859832
2026-05-19     NG   2.502811
2026-05-21     NG   2.988673
2026-05-21     CL  76.359693
2026-05-22     CL  84.626354
2026-05-22     NG   2.015269


* A stale price (same value repeated across consecutive days)


In [47]:
# first set the date to NaN
df.loc[[pd.Timestamp('2026-05-14')], ['close']] = None
# then create a sub-dataframe of just consecutive days and ffill the close 
# column
df.loc[
    [pd.Timestamp('2026-05-13'), pd.Timestamp('2026-05-14')], 
    ['close']] = df.loc[
                        [pd.Timestamp('2026-05-13'), pd.Timestamp('2026-05-14')], 
                        ['ticker', 'close']].groupby(by='ticker').ffill()
print(df)


           ticker      close
date                        
2026-05-08     CL  82.835803
2026-05-08     NG   2.812686
2026-05-11     CL  79.417698
2026-05-11     NG   2.290038
2026-05-12     CL  81.152721
2026-05-12     NG   2.110996
2026-05-13     CL  76.231733
2026-05-13     NG   2.462420
2026-05-14     CL  76.231733
2026-05-14     NG   2.462420
2026-05-15     CL  80.390492
2026-05-15     NG   2.835200
2026-05-18     CL        NaN
2026-05-18     NG        NaN
2026-05-19     CL  80.859832
2026-05-19     NG   2.502811
2026-05-21     NG   2.988673
2026-05-21     CL  76.359693
2026-05-22     CL  84.626354
2026-05-22     NG   2.015269


* An obvious bad print (price that's clearly wrong — use your judgment on what that looks like)

In [48]:
# update Nat Gas
ng_mask = (df.index == pd.Timestamp('2026-05-21')) & (df['ticker'] == 'NG')
df.loc[ng_mask,['close']] = df.loc[ng_mask,['close']]*100
# update Crude
cl_mask = (df.index == pd.Timestamp('2026-05-21')) & (df['ticker'] == 'CL')
df.loc[cl_mask,['close']] = df.loc[cl_mask,['close']]*100
print(df)

           ticker        close
date                          
2026-05-08     CL    82.835803
2026-05-08     NG     2.812686
2026-05-11     CL    79.417698
2026-05-11     NG     2.290038
2026-05-12     CL    81.152721
2026-05-12     NG     2.110996
2026-05-13     CL    76.231733
2026-05-13     NG     2.462420
2026-05-14     CL    76.231733
2026-05-14     NG     2.462420
2026-05-15     CL    80.390492
2026-05-15     NG     2.835200
2026-05-18     CL          NaN
2026-05-18     NG          NaN
2026-05-19     CL    80.859832
2026-05-19     NG     2.502811
2026-05-21     NG   298.867337
2026-05-21     CL  7635.969292
2026-05-22     CL    84.626354
2026-05-22     NG     2.015269



* Set a datetime index

In [52]:
print(df.index.dtype)

datetime64[us]


In [ ]:
detect_stale_prices(df)